# Overfit ACS 2018 target and four-feature membership attack
Training size and settings are defined below; the existing target configuration is preserved. Target training uses a fixed epoch budget without early stopping or best-checkpoint restoration. Preprocessing fits only on actual target training records. Designated MIA members stay in training; held-out target records never enter training.

The restored attack matches experiments_1.ipynb: five diverse Adult shadows with their original default settings, class-scaled logit, blind walk, entropy, cross-entropy, and a fixed random forest attack. Auxiliary validation alone selects thresholds. This is the earlier exploratory configuration, not a guarantee of reproducing a particular accuracy. No experiment was run during restoration.


In [1]:
import importlib
import pandas as pd
import src.data
import src.training
importlib.reload(src.data)
importlib.reload(src.training)
from src.data import load_acs_income, split_acs_data, subset_target_training, preprocess_partitions
from src.training import train_target_model, evaluate_model

OVERFIT_SEED = 42
OVERFIT_TRAIN_SIZE = 145_000
SPLIT_CONFIG = dict(validation_size=10_000, test_size=10_000,
                   mia_nonmember_size=10_000, mia_member_size=10_000, seed=OVERFIT_SEED)
OVERFIT_CONFIG = dict(
    hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0,
    seed=OVERFIT_SEED, device='cpu', early_stopping=False,
    restore_best=False, monitor_train=True,
)

In [2]:
overfit_source = load_acs_income('data/processed/acs_income_ca_2018.csv')
baseline_partitions_for_ids = split_acs_data(overfit_source, **SPLIT_CONFIG)
overfit_partitions = subset_target_training(
    baseline_partitions_for_ids, train_size=OVERFIT_TRAIN_SIZE, seed=OVERFIT_SEED)
print('Unused baseline training records:',
      len(baseline_partitions_for_ids['train']) - len(overfit_partitions['train']))
pd.DataFrame({name: {'records': len(frame), 'positive_rate': frame['income_gt_50k'].mean()}
              for name, frame in overfit_partitions.items()}).T

Unused baseline training records: 15665


,records,positive_rate
validation,10000.0,0.410600
test,10000.0,0.410600
mia_nonmembers,10000.0,0.410600
train,150000.0,0.410567
mia_members,10000.0,0.410600


In [3]:
# Fit ONLY on the reduced training set. Do not reuse the baseline encoder.
overfit_preprocessor, overfit_arrays = preprocess_partitions(overfit_partitions)
{name: values['X'].shape for name, values in overfit_arrays.items()}

{'validation': (10000, 815),
 'test': (10000, 815),
 'mia_nonmembers': (10000, 815),
 'train': (150000, 815),
 'mia_members': (10000, 815)}

In [4]:
# This cell trains when YOU run it. A large CPU model may take time.
overfit_model, overfit_history, overfit_metadata = train_target_model(
    overfit_arrays['train']['X'], overfit_arrays['train']['y'],
    overfit_arrays['validation']['X'], overfit_arrays['validation']['y'],
    **OVERFIT_CONFIG)
print('Returned epoch:', overfit_metadata['returned_epoch'])
pd.DataFrame(overfit_history)

Epoch 001: train loss=0.3998, validation loss=0.3783, validation accuracy=0.8321
Epoch 002: train loss=0.3743, validation loss=0.3763, validation accuracy=0.8321
Epoch 003: train loss=0.3619, validation loss=0.3786, validation accuracy=0.8297
Epoch 004: train loss=0.3454, validation loss=0.3884, validation accuracy=0.8277
Epoch 005: train loss=0.3235, validation loss=0.4004, validation accuracy=0.8243
Epoch 006: train loss=0.2978, validation loss=0.4364, validation accuracy=0.8234
Epoch 007: train loss=0.2703, validation loss=0.4693, validation accuracy=0.8176
Epoch 008: train loss=0.2441, validation loss=0.5555, validation accuracy=0.8105
Epoch 009: train loss=0.2212, validation loss=0.6603, validation accuracy=0.8107
Epoch 010: train loss=0.2013, validation loss=0.6800, validation accuracy=0.8031
Epoch 011: train loss=0.1836, validation loss=0.7865, validation accuracy=0.8036
Epoch 012: train loss=0.1692, validation loss=0.8969, validation accuracy=0.8002
Epoch 013: train loss=0.1556

,epoch,train_loss,validation_accuracy,validation_roc_auc,validation_log_loss,train_eval_accuracy,train_eval_roc_auc,train_eval_log_loss
0,1,0.399777,0.8321,0.908157,0.378270,0.831667,0.911236,0.371572
1,2,0.374264,0.8321,0.908649,0.376318,0.839613,0.918652,0.357803
2,3,0.361924,0.8297,0.907316,0.378575,0.848700,0.925847,0.341631
3,4,0.345438,0.8277,0.906243,0.388371,0.860007,0.936408,0.317132
4,5,0.323547,0.8243,0.899850,0.400419,0.873133,0.947969,0.293944
...,...,...,...,...,...,...,...,...
95,96,0.028257,0.7864,0.854881,1.888207,0.989653,0.999664,0.023318
96,97,0.027682,0.7882,0.853513,1.883533,0.989193,0.999593,0.025590
97,98,0.029767,0.7879,0.854481,1.901641,0.988760,0.999601,0.025966
98,99,0.028266,0.7911,0.854817,1.866413,0.988293,0.999510,0.028428


## Inspect overfitting
Compare end-of-epoch training accuracy/loss against validation. train_loss is averaged across updates during the epoch; train_eval_log_loss evaluates the final weights for that epoch, so it is the directly comparable loss. A growing gap supports overfitting; a high training accuracy alone does not.

In [5]:
overfit_learning_curve = pd.DataFrame(overfit_history)
overfit_learning_curve['accuracy_gap_pp'] = 100 * (
    overfit_learning_curve['train_eval_accuracy'] - overfit_learning_curve['validation_accuracy'])
overfit_learning_curve['validation_minus_train_loss'] = (
    overfit_learning_curve['validation_log_loss'] - overfit_learning_curve['train_eval_log_loss'])
overfit_learning_curve[['epoch', 'train_eval_accuracy', 'validation_accuracy',
    'accuracy_gap_pp', 'train_eval_log_loss', 'validation_log_loss',
    'validation_minus_train_loss']].iloc[::10].copy()

,epoch,train_eval_accuracy,validation_accuracy,accuracy_gap_pp,train_eval_log_loss,validation_log_loss,validation_minus_train_loss
0,1,0.831667,0.8321,-0.043333,0.371572,0.378270,0.006698
10,11,0.931713,0.8036,12.811333,0.159082,0.786466,0.627384
20,21,0.967353,0.7920,17.535333,0.077191,1.311392,1.234201
30,31,0.975793,0.7886,18.719333,0.056767,1.507580,1.450813
40,41,0.981187,0.7891,19.208667,0.045041,1.661348,1.616307
50,51,0.984653,0.7927,19.195333,0.035799,1.759042,1.723243
60,61,0.985627,0.7885,19.712667,0.033795,1.787340,1.753544
70,71,0.987113,0.7862,20.091333,0.029774,1.846422,1.816648
80,81,0.987720,0.7882,19.952000,0.028305,1.866463,1.838158
90,91,0.989000,0.7902,19.880000,0.024868,1.877895,1.853027


## Final income-prediction evaluation
Evaluate after the fixed training budget. Do not choose epochs/settings using test results. The membership attack below uses this target and its own fitted preprocessor.


In [6]:
overfit_metrics = {name: evaluate_model(overfit_model, values['X'], values['y'])
    for name, values in overfit_arrays.items() if name in ['train', 'validation', 'test']}
display(pd.DataFrame(overfit_metrics).T)
print('Train-validation accuracy gap (percentage points):',
      100 * (overfit_metrics['train']['accuracy'] - overfit_metrics['validation']['accuracy']))
print('Train-test accuracy gap (percentage points):',
      100 * (overfit_metrics['train']['accuracy'] - overfit_metrics['test']['accuracy']))

,accuracy,roc_auc,log_loss
validation,0.788100,0.856384,1.941775
test,0.795600,0.861833,1.877053
train,0.990347,0.999717,0.021427


Train-validation accuracy gap (percentage points): 20.224666666666668
Train-test accuracy gap (percentage points): 19.474666666666675


## Random forest membership attack: complete-feature baseline
Run after training the overfit 2018 target. This uses the same attack procedure as the overfit notebook: five diverse Adult shadows with original default settings, four paper-inspired features (class-scaled logits, blind walk, entropy, cross-entropy), and a random forest membership classifier.

The attack is fitted on Adult shadow outputs only. Adult attack rows are split by record ID; thresholds are selected on Adult validation only. Feature extraction samples 2,000 members and 2,000 non-members per shadow. Target evaluation uses all 10,000 overfit target MIA members and 10,000 non-members, with this target's own fitted preprocessor. This notebook trains its own attack and reports only its own target result; no cross-notebook results are reused or compared.

Walk defaults: 20 steps, two walks, categorical change probability .05, batch size 512, seed 42. Repeated model queries can take time. The tabular walk is an adaptation, not an exact reproduction; see data/paper_attack_features.md. True income labels are assumed known. No target MIA results select settings. Run cells manually; none were executed during this edit.

In [27]:
import importlib
import src.shadow_models
import src.membership_attack
import src.paper_attack_features
importlib.reload(src.shadow_models)
importlib.reload(src.membership_attack)
importlib.reload(src.paper_attack_features)
from src.shadow_models import load_adult_data, train_shadow_models, summarize_shadow_models
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import collect_shadow_paper_features, collect_target_paper_features

WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=42)
SHADOW_RECORDS_PER_CLASS = 2000
# Train the five Adult shadows when you run this cell.
# Skip this cell if shadow_results is already loaded and these imports/settings are set.
adult_auxiliary = load_adult_data()
shadow_results = train_shadow_models(adult_auxiliary, seed=42)
summarize_shadow_models(shadow_results)

Training shadow 1/5: xgboost
Training shadow 2/5: random_forest
Training shadow 3/5: mlp
Epoch 001: train loss=0.4162, validation loss=0.3245, validation accuracy=0.8483
Epoch 002: train loss=0.3166, validation loss=0.3166, validation accuracy=0.8531
Epoch 003: train loss=0.3112, validation loss=0.3153, validation accuracy=0.8525
Epoch 004: train loss=0.3074, validation loss=0.3121, validation accuracy=0.8572
Epoch 005: train loss=0.3066, validation loss=0.3150, validation accuracy=0.8513
Epoch 006: train loss=0.3031, validation loss=0.3133, validation accuracy=0.8551
Epoch 007: train loss=0.3022, validation loss=0.3122, validation accuracy=0.8565
Epoch 008: train loss=0.3013, validation loss=0.3117, validation accuracy=0.8559
Epoch 009: train loss=0.2997, validation loss=0.3118, validation accuracy=0.8586
Epoch 010: train loss=0.2982, validation loss=0.3111, validation accuracy=0.8552
Epoch 011: train loss=0.2978, validation loss=0.3149, validation accuracy=0.8543
Epoch 012: train los

records  accuracy   roc_auc  log_loss
shadow_model        partition                                        
xgboost             train         19536  0.883241  0.941862  0.254841
                    validation     9769  0.875832  0.927968  0.277922
                    nonmembers    19537  0.871577  0.926679  0.280469
random_forest       train         19536  0.894656  0.958916  0.246325
                    validation     9769  0.865800  0.921603  0.298265
                    nonmembers    19537  0.865281  0.916223  0.305339
mlp                 train         19536  0.862664  0.920766  0.294177
                    validation     9769  0.855154  0.911535  0.311146
                    nonmembers    19537  0.857399  0.913891  0.306424
decision_tree       train         19536  0.879863  0.945487  0.238998
                    validation     9769  0.846351  0.865680  1.032059
                    nonmembers    19537  0.851871  0.871910  0.999966
logistic_regression train         19536  0.856214  0.909753  0.314372
                    validation     9769  0.849729  0.903692  0.322259
                    nonmembers    19537  0.849619  0.903744  0.321650

In [28]:
# Extract the four features by querying this notebook's fitted shadows.
adult_for_rf_queries = load_adult_data()
shadow_attack_data, rf_shadow_walk_diagnostics = collect_shadow_paper_features(
    shadow_results, adult_for_rf_queries, per_class=SHADOW_RECORDS_PER_CLASS,
    config=WALK_CONFIG, seed=42)
display(pd.DataFrame(rf_shadow_walk_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=42, algorithm='random_forest', feature_set='paper')
# Adult attack diagnostics, not target membership performance.
pd.DataFrame(attack_result['metrics']).T

Attack feature extraction: 512/4,000
Attack feature extraction: 1,024/4,000
Attack feature extraction: 1,536/4,000
Attack feature extraction: 2,048/4,000
Attack feature extraction: 2,560/4,000
Attack feature extraction: 3,072/4,000
Attack feature extraction: 3,584/4,000
Attack feature extraction: 4,000/4,000
Attack feature extraction: 512/4,000
Attack feature extraction: 1,024/4,000
Attack feature extraction: 1,536/4,000
Attack feature extraction: 2,048/4,000
Attack feature extraction: 2,560/4,000
Attack feature extraction: 3,072/4,000
Attack feature extraction: 3,584/4,000
Attack feature extraction: 4,000/4,000
Attack feature extraction: 512/4,000
Attack feature extraction: 1,024/4,000
Attack feature extraction: 1,536/4,000
Attack feature extraction: 2,048/4,000
Attack feature extraction: 2,560/4,000
Attack feature extraction: 3,072/4,000
Attack feature extraction: 3,584/4,000
Attack feature extraction: 4,000/4,000
Attack feature extraction: 512/4,000
Attack feature extraction: 1,024/

,record_queries,max_record_queries,walk_censored_fraction,blind_walk_enabled,config
xgboost,130632,164000,0.685625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest,138296,164000,0.769,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp,132415,164000,0.692375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree,117573,164000,0.549625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression,132848,164000,0.695375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,11984.0,5958.0,0.614403,0.614635,0.663945,0.049681,0.5004,3457.0,2569.0,2052.0,3906.0,0.624076,0.020309,0.002323
validation,4023.0,2032.0,0.515536,0.515057,0.510761,0.009350,0.5004,932.0,1059.0,890.0,1142.0,0.624076,0.009350,0.009543
test,3993.0,2010.0,0.517155,0.516845,0.520103,0.004478,0.5004,934.0,1049.0,879.0,1131.0,0.624076,0.006468,0.010590


In [29]:
# Query this notebook's overfit target using its own fitted preprocessor.
target_attack_data, rf_target_walk_diagnostics = collect_target_paper_features(
    overfit_model, overfit_preprocessor, overfit_partitions, config=WALK_CONFIG, seed=42)
display(pd.Series(rf_target_walk_diagnostics, name='target query diagnostics'))
target_attack_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
pd.DataFrame({
    'Adult attack test': attack_result['metrics']['test'],
    'Overfit ACS 2018 target MIA': target_attack_metrics,
}).T

Attack feature extraction: 512/20,000
Attack feature extraction: 1,024/20,000
Attack feature extraction: 1,536/20,000
Attack feature extraction: 2,048/20,000
Attack feature extraction: 2,560/20,000
Attack feature extraction: 3,072/20,000
Attack feature extraction: 3,584/20,000
Attack feature extraction: 4,096/20,000
Attack feature extraction: 4,608/20,000
Attack feature extraction: 5,120/20,000
Attack feature extraction: 5,632/20,000
Attack feature extraction: 6,144/20,000
Attack feature extraction: 6,656/20,000
Attack feature extraction: 7,168/20,000
Attack feature extraction: 7,680/20,000
Attack feature extraction: 8,192/20,000
Attack feature extraction: 8,704/20,000
Attack feature extraction: 9,216/20,000
Attack feature extraction: 9,728/20,000
Attack feature extraction: 10,240/20,000
Attack feature extraction: 10,752/20,000
Attack feature extraction: 11,264/20,000
Attack feature extraction: 11,776/20,000
Attack feature extraction: 12,288/20,000
Attack feature extraction: 12,800/20,

record_queries                                                       538957
max_record_queries                                                   820000
walk_censored_fraction                                             0.479175
blind_walk_enabled                                                     True
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
Adult attack test,3993.0,2010.0,0.517155,0.516845,0.520103,0.004478,0.5004,934.0,1049.0,879.0,1131.0,0.624076,0.006468,0.01059
Overfit ACS 2018 target MIA,20000.0,10000.0,0.603350,0.603350,0.616628,0.009400,0.5004,2621.0,7379.0,554.0,9446.0,0.624076,0.077600,0.04600
